# Background Tasks and Responsive Interfaces

You will run background JavaFX tasks while keeping the interface responsive and handling every completion path.

CSC-239 · Module 13 · Lesson 4 of 4

A campus staff member will start a controlled background job, keep using its window while the job waits, and handle success, failure or cancellation. This builds on the thread ownership and queue coordination you have studied. [The Module 13 glossary](terms.md) collects the terms explained here.

## Learning Goals

- Run a controlled Task in a background Thread while the native interface still responds to user actions.
- Display success or failure, cancel pending work, and create a fresh task after closing and reopening the example.

## Why This Matters

A window must keep receiving input while background work waits for data. The user also needs useful feedback if work succeeds, fails, or is canceled. Clear ownership ties an unfinished worker to the window that requested it. These rules prepare a responsive interface without claiming that a label change alone proves a worker has stopped.

A responsive interface lets someone keep checking or canceling work instead of wondering whether the application stopped responding. The queue supplies the handoff from an event handler to the worker; the Task supplies the return path for the result or problem. Keeping those paths separate makes ownership and cleanup easier to explain and test.

## Check Your Starting Point

Recall how an event handler changes a JavaFX control, how a lambda refers to surrounding values, and why a potentially waiting operation must run outside an event handler. Distinguish the worker’s result from UI state.

In [ ]:
Your response:

Event handler and live controls:


Captured surrounding values:


Waiting work and UI availability:


Worker result versus UI state:


<details>
<summary>Show answer</summary>

<a id="starting-point-interpretation"></a>

The event handler runs on the UI thread and can update its controls. Background work supplies a result through Task; the UI thread must remain available for events.

A lambda can refer to the surrounding task, queue and controls through their captured references. It still runs on the thread that invokes its callback; lambda syntax alone does not create background execution.

</details>

## Video Demonstration

Follow a waiting background task while the window keeps responding, then compare its completion and cancellation paths.

<video controls preload="metadata" width="960" style="max-width:100%;height:auto;" aria-label="Background JavaFX tasks demonstration">
<source src="media/04_background_tasks_and_responsive_interfaces/demo.mp4" type="video/mp4">
<track kind="captions" src="media/04_background_tasks_and_responsive_interfaces/captions.vtt" srclang="en" label="English">
</video>

[Read the background-task video transcript.](media/04_background_tasks_and_responsive_interfaces/transcript.md)

## Concept

### Separate background work from the live interface

A campus staff member needs to start a background job and keep using its window while the job is pending. Our demonstration uses a controlled count task: it waits for permission, then produces the fixed result six. That deliberate wait gives us time to click another button and check that the interface still responds. It models a pending job; it does not count a real data set or measure processing speed.

Module 12 introduced the **JavaFX Application Thread**, which owns updates to live controls. A long wait on that thread would also delay its button callbacks and display updates. We therefore place the waiting work on a separate worker and keep interface callbacks short.

A JavaFX **Task** represents a background operation and exposes its result and completion state to the interface. `ControlledCountTask` extends `Task<Integer>`, so its successful result has type `Integer`. The task supplies this work method:

```java
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        return 6;
    }
```

This fragment belongs inside the task class. `@Override` checks that the method implements the inherited operation. The keyword **`protected`** permits access from code in the same package and from eligible subclass code. This override keeps the access level of the Task extension method; it does not make the method synchronized. `throws Exception` allows a problem from the background operation to be reported to the task's failure handling.

The first statement waits for a permission item. After that wait, the method checks cancellation, checks whether this test requested a deliberate failure, and otherwise returns six. We will explain those outcomes separately below. None of these statements changes a live control.

A Task can be passed to a `Thread` because it also supplies the Runnable behavior that a thread can execute. Starting that thread causes the task to run its work. Constructing the Task alone does not start the operation.

### Keep the button callback from waiting

The task and interface share a capacity-one queue used as a permission signal. Its item value is `1`, but the presence of an item is what matters: `call` removes it with `take` and can then proceed. The queue is not carrying the result six.

```java
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
```

This fragment installs the callback for **Allow finish** after the queue and button exist. The queue's **`offer`** method tries to add the permission item without waiting for space. It returns `true` if insertion succeeds and `false` if the bounded queue has no room. Only a successful insertion causes this callback to disable the button.

This is a **nonblocking handoff**: the interface sends a signal without waiting for the worker to finish. The worker may wait in `take`; the interface callback uses `offer` so it can return promptly. We also avoid putting `join` or a blocking wait for an unfinished task result in a button callback.

The **Check response** button uses a separate counter owned by the interface thread. Each click updates its label. Clicking it while the task is waiting lets us observe that the interface can still handle another action. We are testing responsiveness during a controlled wait, not claiming the tiny calculation itself takes a noticeable amount of time.

### Show a successful result on the interface thread

When `call` returns normally and the task succeeds, JavaFX delivers its success callback on the JavaFX Application Thread. That callback is the appropriate place to update the live result label:

```java
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
```

The **`getValue`** method reads the task's available result. Here the success callback runs after successful completion, so it can display the returned six as `Result: 6`. This is reading a completed value, not blocking the interface until an unfinished computation produces one.

The callback then disables **Allow finish** and **Cancel task** because this execution no longer needs either action. The background method supplies a value; the interface callback decides how to present it. Keeping that boundary clear lets the computation change later without moving control updates onto a background thread.

### Report a failure as a failure

Background work can fail without producing a valid result. Our class includes a `fail` flag solely to make that path reproducible. With the flag set to `true`, allowing the task to finish reaches the statement that throws `IllegalStateException` with the message `Controlled failure.`

JavaFX stores the problem and invokes the failure callback on the interface thread:

```java
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
```

The **`getException`** method provides the exception associated with the failed task. Its `getMessage` supplies the explanation displayed after `Failed: `. In this controlled case, the complete label becomes `Failed: Controlled failure.`

Displaying that message keeps failure distinct from success with an empty or invented result. The callback also disables actions that no longer apply to this completed execution. The later failure exercise starts a fresh task configured for this outcome, then restores the successful configuration for another fresh run.

### Request cancellation and let the work respond

The **Cancel task** button calls the task's cancellation operation:

```java
    cancel.setOnAction(event -> task.cancel());
```

Task cancellation can request interruption of its running worker. Our pending task is waiting in an interruptible queue operation, so the request can cause that wait to leave by reporting interruption. The task also checks **`isCancelled`** after permission is received, so it can avoid continuing ordinary work when cancellation has already been requested.

This is cooperative cancellation. A program that ignores interruption or spends a long time without checking cancellation can continue executing after a cancellation request. The canceled state is therefore not, by itself, proof that the worker thread has terminated.

In this class, `return 0` after an observed cancellation is an exit path for the work method. It is not a successful zero-count report: cancellation has its own callback, which displays `Canceled` and disables the finish and cancel buttons. Students should distinguish the task's canceled outcome from the successful `Result: 6` outcome.

Our controlled exercise requests cancellation while the task is waiting for permission. It does not demonstrate cancellation halfway through the immediate return of six. A later word-processing task checks cancellation during its loop, applying the same principle to work with repeated steps.

### Tie the worker's lifetime to its window

The window owns this task session. If the user closes the window while the task is pending, its background wait should not continue indefinitely without an interface to manage it. The program registers a cleanup callback:

```java
    stage.setOnHidden(event -> task.cancel());
```

When the stage is hidden, this callback requests cancellation of its task. Closing the native window follows that cleanup path. The callback requests shutdown; it does not block the interface thread by joining the worker there.

The worker is named `course-count-worker` when constructed. The name helps identify this tutorial's thread during the separate cleanup check. It does not give the thread special scheduling behavior.

A closed window proves that the window is gone. A `Canceled` label proves that the interface displayed that task state. Neither observation alone proves that the background thread ended. Establishing that no live thread with the tutorial's diagnostic name remains requires a separate thread check; the window and status label do not supply that evidence. Keeping those observations separate prevents a visually finished interface from hiding unfinished background work.

### Create a fresh task for each session

A Task has a **single-use lifecycle**: one instance represents one execution. A Thread object likewise can be started only once. The **Start task** callback therefore disables its own button before requesting execution:

```java
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
```

The first statement prevents a second ordinary click from trying to start the same thread again. The next statements enable the two pending-task actions and describe the current session. Finally, `worker.start()` requests background execution and allows the callback to return. Calling `worker.run()` directly here would instead execute the waiting work on the interface thread and could freeze the window.

To try another outcome, close the existing window and rerun the complete example. That creates a new task, thread, permission queue, click counter, and controls. Re-enabling Start on the old task would not create a fresh execution.

The notebook uses the supplied `Fx` helper from Module 12 to construct and manage the JavaFX window on the proper thread. The window appears in the Workspace desktop view, not as an inline notebook control. Keep the helper setup and complete window program together when following the lesson. The video uses a standalone JavaFX application entry point, but the division between background work and interface callbacks remains the same.

### Use the supplied window helper

The supplied Fx helper opens and closes this notebook’s windows on the appropriate JavaFX thread. Use it as provided; its implementation is not a new concurrency topic in this lesson. Run the setup cell once in this notebook session before the window examples. The later complete programs call Fx.closeWindows before opening their own window. When an exercise asks for a fresh session, rerun its complete program; do not try to restart the old Task.

The window appears in Desktop. Task ready in the notebook means the program created its window; it is not the background result. Use the named window buttons to advance the controlled wait.

In [ ]:
import javafx.application.Platform;
import javafx.stage.Window;
import java.util.ArrayList;
import java.util.concurrent.CountDownLatch;
import java.util.concurrent.TimeUnit;
import java.util.concurrent.atomic.AtomicReference;
class Fx {
    static void run(Runnable action) throws InterruptedException {
        if (Platform.isFxApplicationThread()) {
            action.run();
            return;
        }
        CountDownLatch done = new CountDownLatch(1);
        AtomicReference<Throwable> failure = new AtomicReference<Throwable>();
        Platform.runLater(() -> {
            try { action.run(); }
            catch (Throwable error) { failure.set(error); }
            finally { done.countDown(); }
        });
        if (!done.await(10, TimeUnit.SECONDS)) {
            throw new IllegalStateException("FX operation timed out; restart the kernel.");
        }
        if (failure.get() != null) { throw new RuntimeException(failure.get()); }
    }
    static void start() throws InterruptedException {
        try { Platform.startup(() -> Platform.setImplicitExit(false)); }
        catch (IllegalStateException alreadyStarted) {
            // This call is also safe when this kernel already started JavaFX.
        }
        run(() -> Platform.setImplicitExit(false));
    }
    static void closeWindows() throws InterruptedException {
        run(() -> {
            for (Window window : new ArrayList<Window>(Window.getWindows())) {
                window.hide();
            }
        });
    }
}
Fx.start();
System.out.println("FX ready");

## Worked Example

The staff member will start the controlled count, click Check response while it waits, and then allow it to return six. The queue carries one permission signal, not the numerical result. Read the complete source and the explained behavior before trying the guided retrieval.

```java
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class ControlledCountTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    public ControlledCountTask(BlockingQueue<Integer> permission, boolean fail) {
        this.permission = permission;
        this.fail = fail;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        return 6;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    ControlledCountTask task = new ControlledCountTask(permission, false);
    Thread worker = new Thread(task, "course-count-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Count");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});
```

The complete window program creates a capacity-one permission queue, one ControlledCountTask, one named Thread and a UI-owned click counter. The task is initially configured with fail set to false. The Start task callback disables its own button and starts the worker; it does not perform the waiting calculation on the JavaFX Application Thread.

While the worker waits for permission, Check response updates its own label. That visible change demonstrates that another UI callback can run during the wait. Allow finish uses offer to send permission. The task then returns six, and the success callback displays Result: 6 while disabling the actions for the completed session. These are distinct observations: UI response during waiting and the later successful result.

The same program provides deliberate failure and cancellation paths for the next exercises. Each new attempt must construct a fresh task and thread. Visible status changes help compare the outcomes, but proving that a background thread has ended requires a separate thread check. Do not treat a success, failure or Canceled label as a worker count.

## Guided Practice

Use the explained program first to retrieve its reasoning, then apply the ideas to distinct completion, modification and debugging tasks.

After each Desktop action sequence, use its observation area for the visible results. A window or label observation does not establish the number of live workers.

Recall the already demonstrated result and explain how the program produces it. Identify the relevant owned or shared state and the rule that allows its final observation. Then run the complete example and preserve this response; this is retrieval, not an unseen prediction.

In [ ]:
Your response:

Recalled result and mechanism:


State ownership:


Rule for observing the final result:


In [ ]:
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class ControlledCountTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    public ControlledCountTask(BlockingQueue<Integer> permission, boolean fail) {
        this.permission = permission;
        this.fail = fail;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        return 6;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    ControlledCountTask task = new ControlledCountTask(permission, false);
    Thread worker = new Thread(task, "course-count-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Count");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});

In the Workspace **Desktop** view, use the window from the preceding complete program:

1. Click **Start task**.
2. Click **Check response**.
3. Click **Allow finish**.
4. Try the already-used **Start task** button again and observe whether it accepts another start.

Record the actual output and native control states from this run. Compare them with the already explained result. Identify any difference without changing your original retrieval response.

In [ ]:
Your response:

Initial notebook output:


Native status and response counter:


Control states:


Comparison with recalled result:


Trace permission from the UI handler through the queue to call, then through the matching terminal handler to the label. Explain which action demonstrates responsiveness and why the displayed result alone does not establish worker termination.

In [ ]:
Your response:

Permission and queue path:


Task result and matching UI callback:


Responsiveness evidence:


Limit of visible state as termination evidence:


<details>
<summary>Show answer</summary>

<a id="worked-interpretation"></a>

Permission travels through the queue to the waiting task; the successful result returns through the Task to a UI-thread callback. Check response supplies visible responsiveness evidence. Thread termination requires a separate check, not inference from a label.

</details>

<details><summary>Show animation: Keep controls responsive during a task</summary><p>Follow the task from its waiting state to a responsive interface and a completed result.</p><p><img src="media/04_background_tasks_and_responsive_interfaces/responsive_task_handoff.gif" alt="Five labeled states show a newly constructed task, the worker waiting for permission, a responsive button counter, a nonblocking permission offer, and Result six. Independent native checks show named worker counts separately from the visible controls." width="960" style="max-width:100%;height:auto;"></p></details>

[View still: Keep controls responsive during a task](media/04_background_tasks_and_responsive_interfaces/responsive_task_handoff_still.png)

### Complete the permission handoff

A staff member starts a background counting task, checks that the interface still responds, then gives the task permission to finish. The permission queue holds one signal.

Replace SEND_WITHOUT_WAITING with the queue operation that returns immediately instead of waiting for room. Explain why this UI handler must stay short and what its boolean result means. Put the complete repaired program in the Java work cell, run it, and use the native Desktop buttons in the listed order.

Read this supplied source before using the separate Java work cell. Keep all other statements unchanged.

```java
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class ControlledCountTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    public ControlledCountTask(BlockingQueue<Integer> permission, boolean fail) {
        this.permission = permission;
        this.fail = fail;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        return 6;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    ControlledCountTask task = new ControlledCountTask(permission, false);
    Thread worker = new Thread(task, "course-count-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.SEND_WITHOUT_WAITING(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Count");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});
```

In [ ]:
Your response:

Chosen operation and boolean meaning:


Why the UI handler stays short:


Predicted native sequence:


In the Workspace **Desktop** view, use the window from the preceding complete program:

1. Start task.
2. Check response twice while the task waits.
3. Allow finish.

Record the waiting status, click counter before allowing finish, and final result. Explain which work happened in the background and which updates happened on the application thread.

In [ ]:
Your response:

Waiting status:


Counter before permission:


Final result:


Background work versus UI updates:


<details>
<summary>Show answer</summary>

<a id="13-04-guided-completion-answer-source-1"></a>

Complete solution:

```java
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class ControlledCountTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    public ControlledCountTask(BlockingQueue<Integer> permission, boolean fail) {
        this.permission = permission;
        this.fail = fail;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        return 6;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    ControlledCountTask task = new ControlledCountTask(permission, false);
    Thread worker = new Thread(task, "course-count-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Count");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});
```

Expected observations:

- Waiting for permission before allowing finish
- UI clicks: 2 while still waiting
- Result: 6 after allowing finish

<a id="13-04-guided-completion-interpretation"></a>

offer attempts to add the permission signal and returns immediately. A true result means the signal was accepted, so the Allow finish button can be disabled. The task waits in its background call method, keeping the interface free to handle Check response. The success handler receives the completed value and updates the Label on the application thread.

</details>

### Change and restore the failure flag

The staff tool needs to report a task failure without freezing its buttons. A controlled failure is enabled so the failure path can be tested without depending on an external service.

Change only the ControlledCountTask constructor flag from false to true. Predict the status after permission is given and explain why the success handler should not supply the result. Run a fresh complete example, start it, check responsiveness, then allow it to finish.

Read this supplied source before using the separate Java work cell. Keep all other statements unchanged.

```java
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class ControlledCountTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    public ControlledCountTask(BlockingQueue<Integer> permission, boolean fail) {
        this.permission = permission;
        this.fail = fail;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        return 6;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    ControlledCountTask task = new ControlledCountTask(permission, false);
    Thread worker = new Thread(task, "course-count-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Count");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});
```

In [ ]:
Your response:

Changed flag:


Predicted terminal status:


Why the success handler should not supply a result:


In [ ]:
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class ControlledCountTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    public ControlledCountTask(BlockingQueue<Integer> permission, boolean fail) {
        this.permission = permission;
        this.fail = fail;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        return 6;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    ControlledCountTask task = new ControlledCountTask(permission, false);
    Thread worker = new Thread(task, "course-count-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Count");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});

In the Workspace **Desktop** view, use the window from the preceding complete program:

1. Start task.
2. Check response once.
3. Allow finish.

Record the responsive waiting state and exact failure message. 

In [ ]:
Your response:

Responsive waiting observation:


Exact failure message:


After recording the failure case, restore false and run a fresh complete program with a new Task and Thread. Start the task and allow it to finish.

In the Workspace **Desktop** view, use the window from the preceding complete program:

1. Start task.
2. Allow finish.

Record the actual restored baseline result and compare it with the earlier worked result. Explain which input you restored and why this run creates fresh work objects.

In [ ]:
Your response:

Restored input:


Actual baseline result:


Comparison and fresh-object explanation:


<details>
<summary>Show answer</summary>

<a id="13-04-guided-modification-answer-source-1"></a>

Complete solution:

```java
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class ControlledCountTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    public ControlledCountTask(BlockingQueue<Integer> permission, boolean fail) {
        this.permission = permission;
        this.fail = fail;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        return 6;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    ControlledCountTask task = new ControlledCountTask(permission, true);
    Thread worker = new Thread(task, "course-count-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Count");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});
```

Expected observations:

- Waiting for permission
- UI clicks: 1
- Failed: Controlled failure.

<a id="13-04-guided-modification-interpretation"></a>

The task throws its controlled exception only after permission arrives. Its failure handler reads the completed exception and displays Failed: Controlled failure. The UI response counter still works while permission is pending. This demonstrates one intentional failure path; it is not evidence of recovery from every possible background exception.

</details>

### Repair the worker launch

A maintainer changes the Start task handler to call worker.run directly. The program is still intended to let the user click Allow finish while background work waits.

Read the faulty program without running it. Trace the execution path from the Start task UI handler to worker.run and the permission.take inside call. Explain why the user cannot rely on the Allow finish handler running while that same UI thread is waiting. Repair only the worker-launch call. Put the complete repaired program into the Java work cell, then test it through the native Desktop.

This is a reading-only faulty program. Trace it without running it; run only your repaired program in the Java work cell.

```java
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class ControlledCountTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    public ControlledCountTask(BlockingQueue<Integer> permission, boolean fail) {
        this.permission = permission;
        this.fail = fail;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        return 6;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    ControlledCountTask task = new ControlledCountTask(permission, false);
    Thread worker = new Thread(task, "course-count-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.run();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Count");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});
```

In [ ]:
Your response:

Faulty call path:


Why the interface could wait on itself:


Single launch-call repair:


In the Workspace **Desktop** view, use the window from the preceding complete program:

1. Start task.
2. Check response twice.
3. Cancel task while waiting.

Record the repaired responsive waiting state and the cancellation status. Explain why the faulty version was only traced and why a Canceled label alone does not prove that background execution ended.

In [ ]:
Your response:

Responsive waiting after repair:


Cancellation status:


Why the faulty version was only traced:


What the status does not prove:


<details>
<summary>Show answer</summary>

<a id="13-04-guided-debugging-answer-source-1"></a>

Complete solution:

```java
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class ControlledCountTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    public ControlledCountTask(BlockingQueue<Integer> permission, boolean fail) {
        this.permission = permission;
        this.fail = fail;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        return 6;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    ControlledCountTask task = new ControlledCountTask(permission, false);
    Thread worker = new Thread(task, "course-count-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Count");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});
```

Expected observations:

- Waiting for permission
- UI clicks: 2
- Canceled

<a id="13-04-guided-debugging-interpretation"></a>

Calling run directly invokes work on the current UI-handler thread. The permission take can then block that thread, preventing normal button event handling. start requests a separate worker execution path. The task can wait there while the interface handles response checks and the permission signal. The supplied event handlers keep live UI changes on the application thread.

</details>

### Compare observed terminal paths

Recall and compare the failure and cancellation paths already observed. Identify which terminal handler changes the status, what happens to controls, and why the status text alone does not prove that the named worker has terminated.

In [ ]:
Your response:

Failure path and controls:


Cancellation path and controls:


Why visible state is not worker-termination proof:


In [ ]:
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class ControlledCountTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    public ControlledCountTask(BlockingQueue<Integer> permission, boolean fail) {
        this.permission = permission;
        this.fail = fail;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        return 6;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    ControlledCountTask task = new ControlledCountTask(permission, true);
    Thread worker = new Thread(task, "course-count-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Count");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});

In the Workspace **Desktop** view, use the window from the preceding complete program:

1. Click **Start task**.
2. Click **Check response**.
3. Click **Allow finish**.

In [ ]:
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class ControlledCountTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    public ControlledCountTask(BlockingQueue<Integer> permission, boolean fail) {
        this.permission = permission;
        this.fail = fail;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        return 6;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    ControlledCountTask task = new ControlledCountTask(permission, false);
    Thread worker = new Thread(task, "course-count-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Count");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});

In the Workspace **Desktop** view, use the window from the preceding complete program:

1. Click **Start task**.
2. Click **Check response**.
3. Click **Cancel task**.

Record actual failure and cancellation states and responsiveness. Compare the paths, and explain why their displayed status is not evidence of a live worker count.

In [ ]:
Your response:

Failure: status and responsiveness:


Cancellation: status and responsiveness:


Comparison and termination-evidence limit:


### Close a waiting window and create a new session

Before the fresh close/reopen run, predict the effects of closing a window while its worker is waiting and then creating new Task and Thread objects. Explain why the old objects cannot supply the new run’s lifecycle.

In [ ]:
Your response:

Closing while waiting:


Fresh reopening:


Why old Task and Thread objects cannot be restarted:


In [ ]:
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class ControlledCountTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    public ControlledCountTask(BlockingQueue<Integer> permission, boolean fail) {
        this.permission = permission;
        this.fail = fail;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        return 6;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    ControlledCountTask task = new ControlledCountTask(permission, false);
    Thread worker = new Thread(task, "course-count-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Count");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});

In the Workspace **Desktop** view, use the window from the preceding complete program:

1. Click **Start task**.
2. Click **Check response**.
3. Close the native window while the task is waiting.
4. Return to the notebook and rerun the preceding complete Java program. Then return to Desktop for its new window.
5. Click **Start task**.
6. Click **Allow finish**.

Record the closed window and the fresh reopened task’s actual result. Explain how the new objects differ from restarting the old Task or Thread, and why observing a closed window alone cannot prove the earlier worker terminated.

In [ ]:
Your response:

Closed-window observation:


Fresh task actual result:


New versus old objects:


Limit of visible closure evidence:


<details>
<summary>Show answer</summary>

<a id="task-terminal-answer-source-1"></a>

### Failure

The true failure flag makes call throw after permission arrives. The failure callback displays Failed: Controlled failure.; it does not invent a result. The response counter can still change during the wait.

```java
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class ControlledCountTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    public ControlledCountTask(BlockingQueue<Integer> permission, boolean fail) {
        this.permission = permission;
        this.fail = fail;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        return 6;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    ControlledCountTask task = new ControlledCountTask(permission, true);
    Thread worker = new Thread(task, "course-count-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Count");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});
```

<a id="task-terminal-answer-source-2"></a>

### Cancellation

Cancel while the task is waiting. The cancellation callback displays Canceled, and the interruptible wait can leave. That visible status remains separate from checking that the named worker actually terminated.

```java
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class ControlledCountTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    public ControlledCountTask(BlockingQueue<Integer> permission, boolean fail) {
        this.permission = permission;
        this.fail = fail;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        return 6;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    ControlledCountTask task = new ControlledCountTask(permission, false);
    Thread worker = new Thread(task, "course-count-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Count");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});
```

<a id="task-terminal-answer-source-3"></a>

### Close and reopen

Closing a waiting window requests cancellation. Rerunning this complete source constructs new work objects and controls. Start the fresh task and allow it to finish; its successful label is Result: 6.

```java
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class ControlledCountTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    public ControlledCountTask(BlockingQueue<Integer> permission, boolean fail) {
        this.permission = permission;
        this.fail = fail;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        return 6;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    ControlledCountTask task = new ControlledCountTask(permission, false);
    Thread worker = new Thread(task, "course-count-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Count");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});
```

<a id="task-terminal-interpretation"></a>

Failure, cancellation and successful completion select different terminal handlers. Closing a waiting window requests cancellation; inspect actual worker termination separately. A later session creates new Task and Thread objects. It does not restart the earlier objects.

<details><summary>Show animation: Separate outcomes from worker cleanup</summary><p>Compare the visible task outcome with the separate worker-cleanup check.</p><p><img src="media/04_background_tasks_and_responsive_interfaces/task_terminal_and_window_cleanup.gif" alt="Six labeled states compare separate success, failure, cancellation, close-while-waiting and fresh-reopen cases. A displayed status or closed window is distinct from the separately checked worker count; only new task and thread objects run again." width="960" style="max-width:100%;height:auto;"></p></details>

[View still: Separate outcomes from worker cleanup](media/04_background_tasks_and_responsive_interfaces/task_terminal_and_window_cleanup_still.png)

</details>

## Independent Practice

Build a complete program that transfers the taught mechanism to the following task. Keep the explained answer closed while planning, implementing and testing.

Build WordLengthTask for a campus label preview. Keep the already taught GUI controls and lifecycle structure. Give the Task an array containing map and book, a permission queue and a false failure flag. After permission arrives, total the string lengths with cancellation checks and return the total. Keep UI changes in the terminal handlers. Before implementing, predict the successful result and explain the ownership of input, result and UI controls. Run the complete program, start the task, check responsiveness while waiting and allow it to finish.

In [ ]:
Your response:

Predicted successful result:


Input ownership:


Background result:


UI controls and terminal callbacks:


Cancellation and fresh-session plan:


In the Workspace **Desktop** view, use the window from the preceding complete program:

1. Click **Start task**.
2. Click **Check response**.
3. Click **Allow finish**.
4. Try the already-used **Start task** button again and observe whether it accepts another start.

Record every actual baseline report and each requested native observation. Compare it with your plan, explain the mechanism, and retain any discrepancy as evidence for a repair.

In [ ]:
Your response:

Initial notebook output:


Waiting and response-counter observations:


Successful status and controls:


Comparison, mechanism and any discrepancy:


### Test five distinct variants

For WordLengthTask, recall and compare the failure and cancellation mechanisms already taught, then predict their behavior for this new word task before running it. Predict what closing a waiting window and creating a fresh task will do. Predict successful results for empty input and for map,map. Keep source and native actions matched to each named case; record these predictions before any variant result.

After testing all five cases, use the grouped observation area for their visible results. A window or label observation does not establish the number of live workers.

Use the next five blank Java work cells in this order. Put a complete program in each cell, and keep the grouped predictions below before running any variant. Each case has its own native action list after its Java cell.

1. **Failure:** Change only the failure flag to true; keep map and book.
2. **Cancellation while waiting:** Keep map, book and a false failure flag; request cancellation before allowing finish.
3. **Close and reopen:** Keep the baseline inputs; close while waiting, then rerun this complete program to create a fresh session.
4. **Empty input:** Use an empty String array and a false failure flag.
5. **Duplicate words:** Use map and map as the two words, with a false failure flag.

In [ ]:
Your response:

1. Failure: predicted state and mechanism:


2. Cancellation: predicted state and mechanism:


3. Close and reopen: predicted states and object lifecycle:


4. Empty input: predicted result and reason:


5. Duplicate words: predicted result and reason:


#### Case 1: Failure

In the Workspace **Desktop** view, use the window from the preceding complete program:

1. Click **Start task**.
2. Click **Check response**.
3. Click **Allow finish**.

#### Case 2: Cancellation while waiting

In the Workspace **Desktop** view, use the window from the preceding complete program:

1. Click **Start task**.
2. Click **Check response**.
3. Click **Cancel task**.

#### Case 3: Close and reopen

In the Workspace **Desktop** view, use the window from the preceding complete program:

1. Click **Start task**.
2. Click **Check response**.
3. Close the native window while the task is waiting.
4. Return to the notebook and rerun the preceding complete Java program. Then return to Desktop for its new window.
5. Click **Start task**.
6. Click **Allow finish**.

#### Case 4: Empty input

In the Workspace **Desktop** view, use the window from the preceding complete program:

1. Click **Start task**.
2. Click **Check response**.
3. Click **Allow finish**.
4. Try the already-used **Start task** button again and observe whether it accepts another start.

#### Case 5: Duplicate words

In the Workspace **Desktop** view, use the window from the preceding complete program:

1. Click **Start task**.
2. Click **Check response**.
3. Click **Allow finish**.
4. Try the already-used **Start task** button again and observe whether it accepts another start.

Record each variant’s actual status and responsiveness separately. Explain how failure, cancellation, fresh reopening and changed word inputs differ. State why these visible outcomes alone do not prove worker termination. Preserve earlier predictions.

In [ ]:
Your response:

1. Failure: actual state, responsiveness and explanation:


2. Cancellation: actual state, responsiveness and explanation:


3. Close and reopen: actual states and fresh-object explanation:


4. Empty input: actual result and explanation:


5. Duplicate words: actual result and explanation:


Why visible outcomes alone do not prove worker termination:


<details>
<summary>Show answer</summary>

<a id="independent-answer-source-1"></a>

### Baseline: map and book

The two lengths are three and four, so the successful label is Result: 7. The total belongs to background call; terminal callbacks own the label updates. Check response still runs during the permission wait.

```java
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class WordLengthTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    private String[] words;
    public WordLengthTask(BlockingQueue<Integer> permission, String[] words, boolean fail) {
        this.permission = permission;
        this.fail = fail;
        this.words = words;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        int total = 0;
        for (String word : words) {
            if (isCancelled()) { return 0; }
            total += word.length();
        }
        return total;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    String[] words = {"map", "book"};
    WordLengthTask task = new WordLengthTask(permission, words, false);
    Thread worker = new Thread(task, "course-word-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Word Length");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});
```

<a id="independent-answer-source-2"></a>

### Failure

The same words are supplied, but the true failure flag throws before the loop. The label is Failed: Controlled failure., not a partial length total.

```java
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class WordLengthTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    private String[] words;
    public WordLengthTask(BlockingQueue<Integer> permission, String[] words, boolean fail) {
        this.permission = permission;
        this.fail = fail;
        this.words = words;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        int total = 0;
        for (String word : words) {
            if (isCancelled()) { return 0; }
            total += word.length();
        }
        return total;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    String[] words = {"map", "book"};
    WordLengthTask task = new WordLengthTask(permission, words, true);
    Thread worker = new Thread(task, "course-word-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Word Length");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});
```

<a id="independent-answer-source-3"></a>

### Cancellation while waiting

Cancel before offering permission. The canceled callback displays Canceled. The early zero-return path is an exit for canceled work, not a successful zero-length report.

```java
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class WordLengthTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    private String[] words;
    public WordLengthTask(BlockingQueue<Integer> permission, String[] words, boolean fail) {
        this.permission = permission;
        this.fail = fail;
        this.words = words;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        int total = 0;
        for (String word : words) {
            if (isCancelled()) { return 0; }
            total += word.length();
        }
        return total;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    String[] words = {"map", "book"};
    WordLengthTask task = new WordLengthTask(permission, words, false);
    Thread worker = new Thread(task, "course-word-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Word Length");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});
```

<a id="independent-answer-source-4"></a>

### Close and reopen

Close while waiting, then rerun this complete source. The new Task, Thread and queue form a new session. Allowing that new task to finish produces Result: 7; the old objects are not restarted.

```java
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class WordLengthTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    private String[] words;
    public WordLengthTask(BlockingQueue<Integer> permission, String[] words, boolean fail) {
        this.permission = permission;
        this.fail = fail;
        this.words = words;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        int total = 0;
        for (String word : words) {
            if (isCancelled()) { return 0; }
            total += word.length();
        }
        return total;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    String[] words = {"map", "book"};
    WordLengthTask task = new WordLengthTask(permission, words, false);
    Thread worker = new Thread(task, "course-word-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Word Length");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});
```

<a id="independent-answer-source-5"></a>

### Empty input

The loop has no words to visit. The initialized total remains zero, giving the successful label Result: 0. This successful zero is distinct from cancellation.

```java
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class WordLengthTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    private String[] words;
    public WordLengthTask(BlockingQueue<Integer> permission, String[] words, boolean fail) {
        this.permission = permission;
        this.fail = fail;
        this.words = words;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        int total = 0;
        for (String word : words) {
            if (isCancelled()) { return 0; }
            total += word.length();
        }
        return total;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    String[] words = {};
    WordLengthTask task = new WordLengthTask(permission, words, false);
    Thread worker = new Thread(task, "course-word-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Word Length");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});
```

<a id="independent-answer-source-6"></a>

### Duplicate words

Both occurrences contribute their lengths. Three plus three gives Result: 6. The loop totals entries; it does not remove duplicates.

```java
import javafx.concurrent.Task;
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import java.util.concurrent.BlockingQueue;
import java.util.concurrent.ArrayBlockingQueue;
class WordLengthTask extends Task<Integer> {
    private BlockingQueue<Integer> permission;
    private boolean fail;
    private String[] words;
    public WordLengthTask(BlockingQueue<Integer> permission, String[] words, boolean fail) {
        this.permission = permission;
        this.fail = fail;
        this.words = words;
    }
    @Override
    protected Integer call() throws Exception {
        permission.take();
        if (isCancelled()) { return 0; }
        if (fail) { throw new IllegalStateException("Controlled failure."); }
        int total = 0;
        for (String word : words) {
            if (isCancelled()) { return 0; }
            total += word.length();
        }
        return total;
    }
}
class ClickCounter {
    private int count;
    public ClickCounter() { count = 0; }
    public void increment() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    BlockingQueue<Integer> permission = new ArrayBlockingQueue<Integer>(1);
    String[] words = {"map", "map"};
    WordLengthTask task = new WordLengthTask(permission, words, false);
    Thread worker = new Thread(task, "course-word-worker");
    ClickCounter clicks = new ClickCounter();
    Label status = new Label("Ready");
    Label response = new Label("UI clicks: 0");
    Button start = new Button("Start task");
    Button finish = new Button("Allow finish");
    Button ping = new Button("Check response");
    Button cancel = new Button("Cancel task");
    finish.setDisable(true);
    cancel.setDisable(true);
    start.setOnAction(event -> {
        start.setDisable(true);
        finish.setDisable(false);
        cancel.setDisable(false);
        status.setText("Waiting for permission");
        worker.start();
    });
    finish.setOnAction(event -> {
        if (permission.offer(1)) { finish.setDisable(true); }
    });
    ping.setOnAction(event -> {
        clicks.increment();
        response.setText("UI clicks: " + clicks.getCount());
    });
    cancel.setOnAction(event -> task.cancel());
    task.setOnSucceeded(event -> {
        status.setText("Result: " + task.getValue());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnFailed(event -> {
        status.setText("Failed: " + task.getException().getMessage());
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    task.setOnCancelled(event -> {
        status.setText("Canceled");
        finish.setDisable(true);
        cancel.setDisable(true);
    });
    stage.setOnHidden(event -> task.cancel());
    VBox root = new VBox(10, status, response, start, finish, ping, cancel);
    root.setPadding(new Insets(20));
    stage.setTitle("Background Word Length");
    stage.setScene(new Scene(root, 440, 350));
    stage.show();
    System.out.println("Task ready");
});
```

<a id="independent-interpretation"></a>

The Task totals the supplied words after permission, while terminal handlers update controls on the UI thread. Empty input returns the initialized total; duplicate words each contribute their length. Cancellation and closing must be checked for actual worker termination, and reopening creates a new Task and Thread.

</details>

## Summary

From memory, explain these lesson ideas in a connected account: Task background computation boundary, Nonblocking UI handoff, Success handoff, Failure handoff, Cooperative task cancellation, Window ownership and cleanup, Single-use Task lifecycle. Include one limit of the example evidence.

In [ ]:
Your response:

Background versus UI work:


Nonblocking permission handoff:


Three terminal paths:


Cancellation and window ownership:


Fresh Task and Thread lifecycle:


<details>
<summary>Show answer</summary>

<a id="summary-retrieval-interpretation"></a>

Task runs waiting work away from live controls. A nonblocking offer lets a UI callback send permission. Success, failure and cancellation have distinct callbacks on the UI thread. Cancellation and window hiding request shutdown; visible states alone do not prove thread termination. Each new session creates a fresh Task and Thread.

</details>

## Reflection

Describe a campus or project task that could use this lesson’s mechanism. Identify the work, owned or shared state, completion rule and one limitation of the analogy. Explain what would fail if the rule were omitted.

In [ ]:
Your response:

Ownership or lifecycle rule to check first:


Concrete reason or example:


Evidence needed beyond the visible result:


Use these ownership and completion rules when examining later project requirements. The project specification, rather than this example, defines its rules.

## Supplemental Reading

- [JavaFX 21 Task](https://openjfx.io/javadoc/21/javafx.graphics/javafx/concurrent/Task.html) explains background call, UI-thread completion callbacks, cancellation and the single-use lifecycle.
- [JavaFX 21 Worker](https://openjfx.io/javadoc/21/javafx.graphics/javafx/concurrent/Worker.html) defines task states, results and exceptions.
- [Java 21 BlockingQueue](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/concurrent/BlockingQueue.html) documents waiting take and nonblocking offer.
- [Java 21 ArrayBlockingQueue](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/concurrent/ArrayBlockingQueue.html) explains the fixed-capacity queue used for permission.
- [Java 21 Thread](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Thread.html) covers starting workers and interruption.
- [JavaFX 21 Window](https://openjfx.io/javadoc/21/javafx.graphics/javafx/stage/Window.html#setOnHidden(javafx.event.EventHandler)) documents the hidden-window callback used for cleanup.
- [JavaFX 21 controls](https://openjfx.io/javadoc/21/javafx.controls/javafx/scene/control/package-summary.html) documents Button and Label.
- [JavaFX 21 layout](https://openjfx.io/javadoc/21/javafx.graphics/javafx/scene/layout/VBox.html) explains VBox placement of the controls.
- [JavaFX 21 Platform](https://openjfx.io/javadoc/21/javafx.graphics/javafx/application/Platform.html) documents application-thread scheduling used inside the supplied Fx helper.